# Module 3 — LLM Generation & RAG / No-RAG

**Project:** RAG Evaluation: Effect of Chunking Strategy and Retrieval Parameters on Answer Quality

## What this notebook does
1. Loads Module 2's `retrieval_results` CSV.
2. Loads a Qwen instruction model.
3. Generates one answer with retrieved context (RAG).
4. Generates a baseline answer without retrieved context (No-RAG).
5. Saves a CSV that Module 4 can score.

**Input expected from Module 2:** `config_id`, `query`, `chunk_size`, `overlap`, `top_k`, `rank`, `paper_id`, `section`, `text`, and `similarity_score`.

Run this notebook in Google Colab with **Runtime → Change runtime type → T4 GPU** if available.

## Cell 1 — Install required packages



In [ ]:

%pip install -q transformers accelerate pandas==2.2.3




## Cell 2 — Check GPU and import libraries

In [ ]:
import os
import gc
import json
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. The model may run slowly on CPU.")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


## Cell 3 — Upload Module 2's CSV output

In Module 2 notebook, export the DataFrame with:

```python
retrieval_results.to_csv("module_2_retrieval_results.csv", index=False)
```


In [ ]:
import zipfile
import os
from google.colab import files

uploaded = files.upload()


# Search for CSV files among uploaded and extracted files
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("No CSV file found. Please upload the Module 2 retrieval-results CSV or a ZIP containing it.")

INPUT_CSV = csv_files[0]
retrieval_results = pd.read_csv(INPUT_CSV)
print("Loaded:", INPUT_CSV)
print("Shape:", retrieval_results.shape)
print("Columns:", retrieval_results.columns.tolist())
display(retrieval_results.head())

Saving module_2_retrieval_results.csv to module_2_retrieval_results.csv
Loaded: module_2_retrieval_results.csv
Shape: (228, 12)
Columns: ['config_id', 'query', 'chunk_size', 'overlap', 'top_k', 'rank', 'window_id', 'parent_chunk_id', 'paper_id', 'section', 'text', 'similarity_score']


,config_id,query,chunk_size,overlap,top_k,rank,window_id,parent_chunk_id,paper_id,section,text,similarity_score
0,CFG_001,What is retrieval-augmented generation?,250,0,1,1,C000119_W001,C000119,P03,II-A Naive RAG,". facing complex issues, a single retrieval ba...",0.819905
1,CFG_001,How does retrieval improve the quality of gene...,250,0,1,1,C000170_W001,C000170,P03,VI-C 1 Quality Scores,"quality scores include context relevance, answ...",0.802780
2,CFG_001,What are the challenges of using retrieval in ...,250,0,1,1,C000129_W001,C000129,P03,III Retrieval,"in the context of rag, it is crucial to effici...",0.872418
3,CFG_002,What is retrieval-augmented generation?,250,0,3,1,C000119_W001,C000119,P03,II-A Naive RAG,". facing complex issues, a single retrieval ba...",0.819905
4,CFG_002,What is retrieval-augmented generation?,250,0,3,2,C000160_W001,C000160,P03,V-A Iterative Retrieval,iterative retrieval is a process where the kno...,0.803401


## Cell 4 — Validate and prepare Module 2 data

This handles the long-format output from Module 2: each row is one retrieved chunk. Rows are grouped by question and experiment configuration to create the context for Qwen.

In [ ]:
required_columns = {
    "query", "chunk_size", "overlap", "top_k", "rank",
    "paper_id", "section", "text", "similarity_score"
}
missing = required_columns - set(retrieval_results.columns)
if missing:
    raise ValueError(
        f"Missing expected Module 2 columns: {sorted(missing)}. "
        "Check that you uploaded retrieval_results, not the original documents CSV."
    )

# Some versions may not contain config_id; create a stable configuration label if absent.
if "config_id" not in retrieval_results.columns:
    retrieval_results["config_id"] = (
        "CH" + retrieval_results["chunk_size"].astype(str)
        + "_OV" + retrieval_results["overlap"].astype(str)
        + "_K" + retrieval_results["top_k"].astype(str)
    )

retrieval_results["rank"] = pd.to_numeric(retrieval_results["rank"], errors="coerce")
retrieval_results["top_k"] = pd.to_numeric(retrieval_results["top_k"], errors="coerce")
retrieval_results["chunk_size"] = pd.to_numeric(retrieval_results["chunk_size"], errors="coerce")
retrieval_results["overlap"] = pd.to_numeric(retrieval_results["overlap"], errors="coerce")
retrieval_results["similarity_score"] = pd.to_numeric(
    retrieval_results["similarity_score"], errors="coerce"
)

retrieval_results["text"] = retrieval_results["text"].fillna("").astype(str)
retrieval_results["query"] = retrieval_results["query"].fillna("").astype(str)

retrieval_results = retrieval_results[
    retrieval_results["query"].str.strip().ne("")
    & retrieval_results["text"].str.strip().ne("")
].copy()

print("Valid retrieval rows:", len(retrieval_results))
print("Unique questions:", retrieval_results["query"].nunique())
print("Configurations:", retrieval_results["config_id"].nunique())
print("Chunk sizes:", sorted(retrieval_results["chunk_size"].dropna().unique().tolist()))
print("Overlaps:", sorted(retrieval_results["overlap"].dropna().unique().tolist()))
print("Top-K values:", sorted(retrieval_results["top_k"].dropna().unique().tolist()))

Valid retrieval rows: 228
Unique questions: 3
Configurations: 16
Chunk sizes: [250, 750]
Overlaps: [0, 200]
Top-K values: [1, 3, 5, 10]


## Cell 5 — Load Qwen

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model_kwargs = {"device_map": "auto"}
if torch.cuda.is_available():
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    **model_kwargs
)
model.eval()

print("Loaded model:", MODEL_NAME)
print("Input device:", model.get_input_embeddings().weight.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded model: Qwen/Qwen2.5-1.5B-Instruct
Input device: cuda:0


## Cell 6 — Define answer-generation functions

The RAG and No-RAG calls use the same model and generation settings. The difference is whether the prompt includes retrieved context.

The RAG prompt instructs the model to rely on the supplied excerpts and to say when the excerpts do not contain enough information. This instruction reduces, but cannot eliminate, hallucinations.

In [ ]:
MAX_NEW_TOKENS = 220
TEMPERATURE = 0.0  # deterministic / greedy-style generation
MAX_INPUT_TOKENS = 6000  # keep context bounded for this model/runtime

def _generate_from_messages(messages, max_new_tokens=MAX_NEW_TOKENS):
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    )
    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer

def generate_no_rag_answer(query):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful research assistant. Answer the question clearly. "
                "Do not invent citations or claim to have read documents you were not given. "
                "If you are unsure, say so."
            )
        },
        {"role": "user", "content": query}
    ]
    return _generate_from_messages(messages)

def generate_rag_answer(query, retrieved_context):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful research-paper question-answering assistant. "
                "Answer using the supplied context. Do not add unsupported claims. "
                "If the context does not contain enough evidence, explicitly say that "
                "the provided excerpts are insufficient. Keep the answer concise."
            )
        },
        {
            "role": "user",
            "content": (
                f"Question:\n{query}\n\n"
                f"Retrieved research-paper excerpts:\n{retrieved_context}\n\n"
                "Answer the question using the excerpts above. If the answer is not supported "
                "by the excerpts, state that clearly."
            )
        }
    ]
    return _generate_from_messages(messages)

print("Generation functions are ready.")

Generation functions are ready.


## Cell 7 — Build context from ranked chunks

In [ ]:
def build_context(group):
    group = group.sort_values("rank", ascending=True)
    parts = []
    for _, row in group.iterrows():
        paper_id = str(row.get("paper_id", "unknown"))
        section = str(row.get("section", "unknown"))
        rank = row.get("rank", "")
        text = str(row.get("text", "")).strip()
        if text:
            parts.append(
                f"[Retrieved chunk rank={rank}; paper_id={paper_id}; section={section}]\n{text}"
            )
    return "\n\n".join(parts)

# Module 2 has one row per retrieved chunk. This creates one group per question/configuration.
group_columns = ["config_id", "query", "chunk_size", "overlap", "top_k"]
experiment_groups = list(
    retrieval_results.groupby(group_columns, dropna=False, sort=False)
)

print("Question/configuration groups to process:", len(experiment_groups))
print("Example group keys:", experiment_groups[0][0] if experiment_groups else "No groups found")

Question/configuration groups to process: 48
Example group keys: ('CFG_001', 'What is retrieval-augmented generation?', np.int64(250), np.int64(0), np.int64(1))


## Cell 8 — Run a small pilot first

In [ ]:
PILOT_GROUPS = min(3, len(experiment_groups))
no_rag_cache = {}
pilot_rows = []

for group_index, (keys, group) in enumerate(experiment_groups[:PILOT_GROUPS], start=1):
    config_id, query, chunk_size, overlap, top_k = keys
    context = build_context(group)

    if query not in no_rag_cache:
        no_rag_cache[query] = generate_no_rag_answer(query)

    rag_answer = generate_rag_answer(query, context)

    pilot_rows.append({
        "config_id": config_id,
        "query": query,
        "chunk_size": chunk_size,
        "overlap": overlap,
        "top_k": top_k,
        "rag_answer": rag_answer,
        "no_rag_answer": no_rag_cache[query],
        "retrieved_context": context
    })
    print(f"Completed pilot {group_index}/{PILOT_GROUPS}")

pilot_df = pd.DataFrame(pilot_rows)
display(pilot_df[["query", "chunk_size", "overlap", "top_k", "rag_answer", "no_rag_answer"]])

Completed pilot 1/3
Completed pilot 2/3
Completed pilot 3/3


,query,chunk_size,overlap,top_k,rag_answer,no_rag_answer
0,What is retrieval-augmented generation?,250,0,1,Retrieval-augmented generation refers to an ap...,Retrieval-Augmented Generation (RAG) is an app...
1,How does retrieval improve the quality of gene...,250,0,1,Retrieval improves the quality of generated an...,Retrieval can significantly enhance the qualit...
2,What are the challenges of using retrieval in ...,250,0,1,The challenges of using retrieval in a RAG (Re...,Retrieval-based approaches (RAG) in natural la...


## Cell 9 — Run the full generation experiment

In [ ]:
all_result_rows = []
no_rag_cache = {}

for group_index, (keys, group) in enumerate(experiment_groups, start=1):
    config_id, query, chunk_size, overlap, top_k = keys
    group = group.sort_values("rank", ascending=True)
    context = build_context(group)

    try:
        if query not in no_rag_cache:
            no_rag_cache[query] = generate_no_rag_answer(query)
        no_rag_answer = no_rag_cache[query]
        rag_answer = generate_rag_answer(query, context)
        status = "success"
        error_message = ""
    except Exception as e:
        rag_answer = ""
        no_rag_answer = no_rag_cache.get(query, "")
        status = "error"
        error_message = repr(e)
        print(f"Error on group {group_index} ({config_id}): {error_message}")

    paper_ids = [
        str(x) for x in group["paper_id"].dropna().drop_duplicates().tolist()
    ]
    chunk_ids = (
        group["parent_chunk_id"].dropna().astype(str).tolist()
        if "parent_chunk_id" in group.columns
        else []
    )
    window_ids = (
        group["window_id"].dropna().astype(str).tolist()
        if "window_id" in group.columns
        else []
    )
    similarity_scores = group["similarity_score"].dropna().tolist()

    all_result_rows.append({
        "config_id": config_id,
        "query": query,
        "chunk_size": chunk_size,
        "overlap": overlap,
        "top_k": top_k,
        "num_retrieved_chunks": len(group),
        "retrieved_paper_ids": json.dumps(paper_ids, ensure_ascii=False),
        "retrieved_parent_chunk_ids": json.dumps(chunk_ids, ensure_ascii=False),
        "retrieved_window_ids": json.dumps(window_ids, ensure_ascii=False),
        "similarity_scores": json.dumps(similarity_scores),
        "retrieved_context": context,
        "rag_answer": rag_answer,
        "no_rag_answer": no_rag_answer,
        "model_name": MODEL_NAME,
        "max_new_tokens": MAX_NEW_TOKENS,
        "generation_mode": "greedy",
        "generation_status": status,
        "error_message": error_message
    })

    if group_index % 5 == 0 or group_index == len(experiment_groups):
        print(f"Processed {group_index}/{len(experiment_groups)} groups")

generation_results = pd.DataFrame(all_result_rows)
print("Generation result shape:", generation_results.shape)
display(generation_results.head())

Processed 5/48 groups
Processed 10/48 groups
Processed 15/48 groups
Processed 20/48 groups
Processed 25/48 groups
Processed 30/48 groups
Processed 35/48 groups
Processed 40/48 groups
Processed 45/48 groups
Processed 48/48 groups
Generation result shape: (48, 18)


,config_id,query,chunk_size,overlap,top_k,num_retrieved_chunks,retrieved_paper_ids,retrieved_parent_chunk_ids,retrieved_window_ids,similarity_scores,retrieved_context,rag_answer,no_rag_answer,model_name,max_new_tokens,generation_mode,generation_status,error_message
0,CFG_001,What is retrieval-augmented generation?,250,0,1,1,"[""P03""]","[""C000119""]","[""C000119_W001""]",[0.8199053406715393],[Retrieved chunk rank=1; paper_id=P03; section...,Retrieval-augmented generation refers to an ap...,Retrieval-Augmented Generation (RAG) is an app...,Qwen/Qwen2.5-1.5B-Instruct,220,greedy,success,
1,CFG_001,How does retrieval improve the quality of gene...,250,0,1,1,"[""P03""]","[""C000170""]","[""C000170_W001""]",[0.8027803301811218],[Retrieved chunk rank=1; paper_id=P03; section...,Retrieval improves the quality of generated an...,Retrieval can significantly enhance the qualit...,Qwen/Qwen2.5-1.5B-Instruct,220,greedy,success,
2,CFG_001,What are the challenges of using retrieval in ...,250,0,1,1,"[""P03""]","[""C000129""]","[""C000129_W001""]",[0.8724180459976196],[Retrieved chunk rank=1; paper_id=P03; section...,The challenges of using retrieval in a RAG (Re...,Retrieval-based approaches (RAG) in natural la...,Qwen/Qwen2.5-1.5B-Instruct,220,greedy,success,
3,CFG_002,What is retrieval-augmented generation?,250,0,3,3,"[""P03""]","[""C000119"", ""C000160"", ""C000115""]","[""C000119_W001"", ""C000160_W001"", ""C000115_W001""]","[0.8199053406715393, 0.8034006953239441, 0.794...",[Retrieved chunk rank=1; paper_id=P03; section...,Retrieval-augmented generation refers to an ap...,Retrieval-Augmented Generation (RAG) is an app...,Qwen/Qwen2.5-1.5B-Instruct,220,greedy,success,
4,CFG_002,How does retrieval improve the quality of gene...,250,0,3,3,"[""P03"", ""P04""]","[""C000170"", ""C000160"", ""C000194""]","[""C000170_W001"", ""C000160_W001"", ""C000194_W001""]","[0.8027803301811218, 0.7928339242935181, 0.788...",[Retrieved chunk rank=1; paper_id=P03; section...,Retrieval improves the quality of generated an...,Retrieval can significantly enhance the qualit...,Qwen/Qwen2.5-1.5B-Instruct,220,greedy,success,


## Cell 10 — Quality checks

In [ ]:
if not generation_results.empty:
    print("Rows:", len(generation_results))
    print("Successful:", int((generation_results["generation_status"] == "success").sum()))
    print("Errors:", int((generation_results["generation_status"] == "error").sum()))
    print("Missing RAG answers:", int(generation_results["rag_answer"].fillna("").str.strip().eq("").sum()))
    print("Missing No-RAG answers:", int(generation_results["no_rag_answer"].fillna("").str.strip().eq("").sum()))
    display(
        generation_results[
            ["query", "chunk_size", "overlap", "top_k", "generation_status"]
        ].head(10)
    )
else:
    print("No generated results. Check the uploaded CSV and rerun the generation cells.")

Rows: 48
Successful: 48
Errors: 0
Missing RAG answers: 0
Missing No-RAG answers: 0


,query,chunk_size,overlap,top_k,generation_status
0,What is retrieval-augmented generation?,250,0,1,success
1,How does retrieval improve the quality of gene...,250,0,1,success
2,What are the challenges of using retrieval in ...,250,0,1,success
3,What is retrieval-augmented generation?,250,0,3,success
4,How does retrieval improve the quality of gene...,250,0,3,success
5,What are the challenges of using retrieval in ...,250,0,3,success
6,What is retrieval-augmented generation?,250,0,5,success
7,How does retrieval improve the quality of gene...,250,0,5,success
8,What are the challenges of using retrieval in ...,250,0,5,success
9,What is retrieval-augmented generation?,250,0,10,success


## Cell 11 — Export Module 3 results for Module 4

In [ ]:
OUTPUT_CSV = "module_3_generation_results.csv"
generation_results.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUTPUT_CSV)

from google.colab import files
files.download(OUTPUT_CSV)

Saved: module_3_generation_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>